***PREPROCESSING — STEP 1: Caricamento dataset***

In [2]:
# Import librerie
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Carico il CSV in un DataFrame
df = pd.read_csv("TicketEstrazione090326.csv")

***PREPROCESSING — STEP 2: Normalizzazione campo area***

In [3]:
# Mapping completo chiave → valore dei campi Area numerici, basato sulla tabella zhc_dropdown_maia
AREA_MAPPING = {
    '1': 'Amministrazione',
    '2': 'Amm-Economato',
    '3': 'Utenti e Ospiti',
    '4': 'CSS',
    '5': 'Personale',
    '6': 'Hardware',
    '7': 'CCE',
    '8': 'CBA DR STP',
}

# Quanti ticket hanno area numerica?
area_numerica_mask = df['area'].astype(str).str.strip().isin(AREA_MAPPING.keys())
print(f"Ticket con area numerica da mappare: {area_numerica_mask.sum():,}")
print(df[area_numerica_mask]['area'].value_counts())

# Applico mapping
df['area'] = df['area'].apply(
    lambda x: AREA_MAPPING.get(str(x).strip(), x) if pd.notna(x) else x
)

print(f"\nDistribuzione area dopo mapping:")
print(df['area'].value_counts(dropna=False))

Ticket con area numerica da mappare: 853
area
6    842
1      4
5      3
4      3
3      1
Name: count, dtype: int64

Distribuzione area dopo mapping:
area
area_personale                     14263
ciclo_passivo                      10993
ciclo_attivo                        9236
area_sanitaria                      8748
NaN                                 5849
rendicontazione_flussi              2859
protocollo_documentale_delibere     2581
area_sistemistica                   1109
sistema381                          1006
Hardware                             842
area_territoriale                    537
protocollo_delibere                  228
business_intelligence                149
Amministrazione                        4
Personale                              3
CSS                                    3
Utenti e Ospiti                        1
flussi_regionali                       1
Name: count, dtype: int64


***PREPROCESSING — STEP 3: Imputazione priorita_iniziale_cliente***

In [4]:
# Se trovo NaN in priorità iniziale = operatore NON ha modificato la priorità
# Il cliente aveva già scelto quella giusta
# priorita_iniziale_cliente == priorita_finale
# Imputo i NaN con priorita_finale

nan_prima = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN prima dell'imputazione: {nan_prima:,} ({nan_prima/len(df)*100:.1f}%)")

df['priorita_iniziale_cliente'] = df['priorita_iniziale_cliente'].fillna(df['priorita_finale'])

nan_dopo = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN dopo l'imputazione: {nan_dopo:,}")

# Distribuzione dopo imputazione
print(f"\nDistribuzione priorita_iniziale_cliente dopo imputazione:")
print(df['priorita_iniziale_cliente'].value_counts().sort_index())

NaN prima dell'imputazione: 51,401 (88.0%)
NaN dopo l'imputazione: 0

Distribuzione priorita_iniziale_cliente dopo imputazione:
priorita_iniziale_cliente
P1    12872
P2    14933
P3    29415
P4     1192
Name: count, dtype: int64


***PREPROCESSING — STEP 4: Pulizia testi***

In [5]:
# import librerie per pulizia testo
from bs4 import BeautifulSoup
import re

# Funzioni di pulizia testo

def pulisci_html(testo):
    """Pulizia base HTML usando BeautifulSoup."""
    # se il valore non è stringa o è vuoto/non significativo restituisco stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    # costruisco un DOM con BeautifulSoup
    soup = BeautifulSoup(testo, "html.parser")
    # ogni tag <br> viene sostituito con un newline
    for br in soup.find_all("br"):
        br.replace_with("\n")
    # per ogni <p> inserisco uno spazio dopo il tag e lo "sciolgo"
    # (unwrap rimuove il tag lasciando il contenuto)
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    # estraggo il testo finale e tolgo spazi iniziali/finali
    return soup.get_text().strip()


def pulisci_blocco_reperibilita(soup):
    """
    Rimuove il blocco 'Orario reperibilità + Telefono' 
    navigando il DOM.
    """
    # prendo tutti i <br> perchè da lì riconosco la struttura del blocco
    br_tags = soup.find_all("br")
    testo = soup.get_text()
    
    # se nel testo compare la stringa di reperibilità (con o senza accento)
    if "Orario reperibilità" in testo or "Orario reperibilita" in testo:
        # verifico che ci siano almeno due <br> (atteso formattazione del blocco)
        if len(br_tags) >= 2:
            second_br = br_tags[1]
            # estraggo tutti i nodi precedenti al secondo <br> (cioè l'intero blocco)
            for elem in list(second_br.previous_siblings):
                elem.extract()
            # estraggo anche il secondo <br> stesso
            second_br.extract()
    return soup


def pulisci_descrizione(testo):
    """
    Pulizia descrizione ticket:
    - Rimuove blocco reperibilità/telefono
    - Rimuove HTML
    """
    # guard clause: input non valido → stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # parse HTML
    soup = BeautifulSoup(testo, "html.parser")
    # elimino l'eventuale blocco di reperibilità
    soup = pulisci_blocco_reperibilita(soup)
    
    # normalizzo i tag <br> e <p> come in pulisci_html
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    
    testo_pulito = soup.get_text()
    # tolgo triple (o più) newline consecutivi -> lascio al massimo due
    testo_pulito = re.sub(r'\n{3,}', '\n\n', testo_pulito)
    return testo_pulito.strip()


def pulisci_conversazione(testo):
    """
    Pulizia conversazione completa:
    - Divide per separatore ---
    - Pulisce ogni messaggio singolarmente
    - Rimuove blocco reperibilità solo dal primo messaggio cliente
    - Ricostruisce la conversazione
    """
    # guard clause
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # i messaggi sono separati dal marcatore esplicito
    messaggi = testo.split('\n---\n')
    messaggi_puliti = []
    
    # itero su tutti i messaggi
    for i, messaggio in enumerate(messaggi):
        # cerco di separare l'autore dal corpo (formato 'AUTORE: testo')
        if ': ' in messaggio:
            autore, corpo = messaggio.split(': ', 1)
        else:
            autore, corpo = '', messaggio
        
        soup = BeautifulSoup(corpo, "html.parser")
        
        # solo se l'autore è CLIENTE rimuovo il blocco reperibilità
        if autore.upper() == 'CLIENTE':
            soup = pulisci_blocco_reperibilita(soup)
        
        # sostituisco <br> e <p> come prima
        for br in soup.find_all("br"):
            br.replace_with("\n")
        for p in soup.find_all("p"):
            p.insert_after(" ")
            p.unwrap()
        
        corpo_pulito = soup.get_text().strip()
        
        # se il messaggio è vuoto dopo la pulizia lo salto
        if not corpo_pulito:
            continue
        
        # ricostruisco la stringa del messaggio con autore se presente
        if autore:
            if autore.upper() == 'CLIENTE':
                autore_norm = 'CLIENTE'
            else:
                autore_norm = 'OPERATORE'
            messaggi_puliti.append(f"{autore_norm}: {corpo_pulito}")
        else:
            messaggi_puliti.append(corpo_pulito)
    
    # riunisco la conversazione con lo stesso separatore iniziale
    return '\n---\n'.join(messaggi_puliti)

# Applica le funzioni
df['oggetto_pulito']        = df['oggetto'].apply(pulisci_html)
df['descrizione_pulita']    = df['descrizione'].apply(pulisci_descrizione)
df['conversazione_pulita']  = df['conversazione'].apply(pulisci_conversazione)

In [6]:
# Analisi lunghezza descrizione
df['desc_n_char'] = df['descrizione_pulita'].str.len()
df['desc_n_parole'] = df['descrizione_pulita'].str.split().str.len()

print("=== STATISTICHE LUNGHEZZA DESCRIZIONE ===")
print(df[['desc_n_char', 'desc_n_parole']].describe().round(1))
print(f"\nDescrizioni vuote o quasi (< 5 parole): {(df['desc_n_parole'] < 5).sum():,}")
print(f"Descrizioni molto corte (< 10 parole):   {(df['desc_n_parole'] < 10).sum():,}")

=== STATISTICHE LUNGHEZZA DESCRIZIONE ===
       desc_n_char  desc_n_parole
count      58412.0        58412.0
mean         477.6           68.2
std          984.2          131.9
min            0.0            0.0
25%          193.0           27.0
50%          297.0           43.0
75%          464.0           68.0
max        65533.0         8673.0

Descrizioni vuote o quasi (< 5 parole): 504
Descrizioni molto corte (< 10 parole):   1,989


***Test***

In [7]:
mask_rep = df['conversazione'].str.contains('reperibilit', case=False, na=False)
print(f"Ticket con blocco reperibilità nella conversazione grezza: {mask_rep.sum():,}")

if mask_rep.sum() > 0:
    idx = df[mask_rep].index[4]
    print("\n=== CONVERSAZIONE GREZZA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione'][:6000])
    print("\n=== CONVERSAZIONE PULITA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione_pulita'][:6000])

Ticket con blocco reperibilità nella conversazione grezza: 26,998

=== CONVERSAZIONE GREZZA (primi 6000 chars) ===
CLIENTE: <strong>Orario reperibilità contatto: </strong><br /><strong>Telefono: </strong><br /><div class="ql-editor"><p>CIAO IVAN,</p><p><br /></p><p>Stiamo cambiando connettività nelle varie strutture adesso presso la Casa di Riposo Villa del Pensionato - Via San Francesco 3 - 47017 Rocca San Casciano (FC) l'indirizzo IP pubblico è il seguente: 79.62.249.228</p><p><br /></p><p> </p><p><br /></p><p>Si segnala l'urgenza</p><p><br /></p><p> </p><p><br /></p><p>Cordiali Saluti</p></div>
---
Mauro Zaltron: <p>Effettuata modifica IP</p><br />
<p>attendo riscontro</p>
---
CLIENTE: <div class="ql-editor"><p>LA STRUTTURA MI HA COMUNICATO CHE FUNZIONANO I PROGRAMMI 1.0 CHE SI UTILLIZZANO TRAMITE DESKTOP REMOTO</p></div>
---
Mauro Zaltron: <div class="mozaik-inner" style="font-family:Arial, Helvetica, sans-serif;font-size:14px;line-height:22.4px;color:#444444;background-color:rgba(

In [8]:
# Visualizzo l'url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[idx, 'url_ticket']}")


URL del ticket: https://maia.zucchettihc.it/index.php?module=Cases&action=DetailView&record=10160380-f462-cbd5-287a-67584f55e30f


***PREPROCESSING — STEP 6: Costruzione testo_input (oggetto_pulito + descrizione_pulita)***

In [ ]:

# STEP 6: Costruzione testo_input
# Concatenazione oggetto_pulito + spazio + descrizione_pulita
# fillna('') per gestire i NaN, strip() per rimuovere spazi iniziali/finali
df['testo_input'] = (df['oggetto_pulito'].fillna('') + ' ' + df['descrizione_pulita'].fillna('')).str.strip()

# Statistiche
testi_vuoti = (df['testo_input'] == '').sum()
lunghezza_media = df['testo_input'].str.split().str.len().mean()

print(f"Testi vuoti (stringa vuota dopo strip): {testi_vuoti:,}")
print(f"Lunghezza media in parole:              {lunghezza_media:.1f}")

print(f"\nEsempio (primo testo_input non vuoto — primi 800 caratteri):")
esempio = df[df['testo_input'] != '']['testo_input'].iloc[4]
print(esempio[:800])

# visualizzo url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[df['testo_input'] != '', 'url_ticket'].iloc[4]}")


Testi vuoti (stringa vuota dopo strip): 0
Lunghezza media in parole:              73.4

Esempio (primo testo_input non vuoto — primi 400 caratteri):
Richiesta formulazione offerta per CIVILIA NEXT per Asp Distretto di Parma Ciao,
 
Vi chiedo cortesemente di formulare un’offerta per CIVILIA NEXT seguendo il modulo allegato.
 
@Ruggero Maggioni in calce il messaggio di Giorgio.
Nell'offerta è necessario specificare che l'acquisto delle firme remote è escluso. Ogni firma ha un costo di 150 euro + IVA (una tantum), a cui si aggiunge un canone annuo di 80 euro + IVA. Su questo punto, ti chiedo gentilmente di allinearti con Ruggero per eventuali dettagli.
 
@TT.commerciale@zucchettihc.it:
Per quanto riguarda la parte relativa alle firme, attendiamo le indicazioni di Ruggero.
 
Grazie,
Antonino
 
 
 

 
Antonino Arcaria Commerciale

Zucchetti Healthcare S.r.l. 
Viale Trento, 56 | 38068 Rovereto (TN)antonino.arcaria@zucchettihc.it+39 049 93363

URL del ticket: https://maia.zucchettihc.it/index

***PREPROCESSING — STEP 7: Features derivate (n_parole, has_urgenza)***

In [10]:
# Contiamo quante parole ha ogni testo_input
df['n_parole'] = df['testo_input'].str.split().str.len()

# --- has_urgenza ---
# True se contiene almeno una keyword di urgenza forte
URGENZA_KW = ['urgente', 'immediato', 'il prima possibile', 'critico']

def calc_has_urgenza(testo):
    if not isinstance(testo, str):
        return False
    testo_lower = testo.lower()
    return any(kw in testo_lower for kw in URGENZA_KW)

# Applica la funzione
df['has_urgenza']      = df['testo_input'].apply(calc_has_urgenza)

# Verifico distribuzione e correlazione con priorità
for col in ['has_urgenza']:
    n_true = df[col].sum()
    pct = n_true / len(df) * 100
    print(f"\n{col}: {n_true:,} ticket ({pct:.1f}%)")
    print(pd.crosstab(df[col], df['priorita_finale'], normalize='index').mul(100).round(1))


has_urgenza: 3,251 ticket (5.6%)
priorita_finale    P1    P2    P3   P4
has_urgenza                           
False            12.2  28.9  56.3  2.7
True             43.5  33.3  22.9  0.3


***PREPROCESSING — STEP 8: Rimozione ticket senza informazione utile***

In [11]:
# Mostriamo la distribuzione prima di filtrare
print("=== Distribuzione lunghezze testo_input ===")
bins   = [0, 3, 5, 10, 20, 50, float('inf')]
labels = ['<=3', '3-5', '5-10', '10-20', '20-50', '50+']
print(pd.cut(df['n_parole'], bins=bins, labels=labels).value_counts().sort_index())

# Scegliamo di rimuovere i ticket con meno di 3 parole
# Sotto questa soglia non c'è abbastanza testo per capire il problema
mask_troppo_corti = df['n_parole'] < 3

print(f"\nTicket da rimuovere (< 3 parole): {mask_troppo_corti.sum():,}")
print(f"Ticket rimanenti:                  {(~mask_troppo_corti).sum():,}")

# Mostriamo qualche esempio di quello che stiamo rimuovendo
print("\n=== Esempi ticket rimossi ===")
print(df[mask_troppo_corti][['testo_input', 'priorita_finale']].head(10).to_string())

# Rimozione
df = df[~mask_troppo_corti].copy()

=== Distribuzione lunghezze testo_input ===
n_parole
<=3         43
3-5        132
5-10       819
10-20     4354
20-50    25975
50+      27089
Name: count, dtype: int64

Ticket da rimuovere (< 3 parole): 27
Ticket rimanenti:                  58,385

=== Esempi ticket rimossi ===
               testo_input priorita_finale
691       PARTENZA CLIENTE              P3
4815    aggiornamento 2025              P3
6145       VERIFICA UTENTI              P3
6207   fattura elettornica              P3
12913     FORMAZIONE SVAMA              P3
13457        aggiornamento              P3
13686  aggiornamento 26.01              P3
14805        ERRORE PAGOPA              P3
15291  aggiornamento 26.01              P3
19108  VERIFICHE VERIFICHE              P3


In [12]:
df.columns

Index(['url_ticket', 'case_number', 'oggetto', 'descrizione',
       'priorita_finale', 'priorita_iniziale_cliente', 'area',
       'tipologia_intervento', 'articolo', 'modulo_sw', 'reparto',
       'data_creazione', 'data_risoluzione', 'conversazione', 'oggetto_pulito',
       'descrizione_pulita', 'conversazione_pulita', 'desc_n_char',
       'desc_n_parole', 'testo_input', 'n_parole', 'has_urgenza'],
      dtype='str')

***Salvataggio Dataframe Pulito***

In [13]:
# STEP 9 — Selezione colonne finali e salvataggio dataset_clean.csv

COLONNE_FINALI = [
    'url_ticket',
    'case_number',
    'testo_input',
    'priorita_finale',
    'priorita_iniziale_cliente',
    'area',
    'articolo',
    'modulo_sw',
    'has_urgenza',
    'n_parole',
    'data_creazione',
]

df_clean = df[COLONNE_FINALI].copy()

print(f"Righe: {len(df_clean):,}")
print(f"Colonne: {list(df_clean.columns)}")
print(f"\nNaN per colonna:")
print(df_clean.isnull().sum())
print(f"\nDistribuzione priorita_finale:")
print(df_clean['priorita_finale'].value_counts().sort_index())

df_clean.to_csv("dataset_clean.csv", index=False, encoding='utf-8-sig')
print("\nSalvato dataset_clean.csv ✅")

Righe: 58,385
Colonne: ['url_ticket', 'case_number', 'testo_input', 'priorita_finale', 'priorita_iniziale_cliente', 'area', 'articolo', 'modulo_sw', 'has_urgenza', 'n_parole', 'data_creazione']

NaN per colonna:
url_ticket                      0
case_number                     0
testo_input                     0
priorita_finale                 0
priorita_iniziale_cliente       0
area                         5847
articolo                     5879
modulo_sw                    5805
has_urgenza                     0
n_parole                        0
data_creazione                  0
dtype: int64

Distribuzione priorita_finale:
priorita_finale
P1     8131
P2    16998
P3    31783
P4     1473
Name: count, dtype: int64

Salvato dataset_clean.csv ✅
